# 01. EDA and Schema Audit

**SW5 Satellite Collision Risk Predictor (NearMiss)**

This notebook verifies the schema, unit ranges, risk floor values, and class imbalance of the official ESA Collision Avoidance Challenge dataset (Zenodo record 4463683).

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.load import load_config, load_raw_cdms

cfg = load_config("config.yaml")
raw_dir = cfg["data"]["raw_dir"]
print("Config loaded successfully. Raw dir:", raw_dir)

## 1. Load Datasets & Check Dimensions

In [2]:
df_train = load_raw_cdms(raw_dir, "train")
df_test = load_raw_cdms(raw_dir, "test")
df_priv = load_raw_cdms(raw_dir, "test_private")

print(f"Train CDMs: {len(df_train):,} across {df_train['event_id'].nunique():,} events. Columns: {len(df_train.columns)}")
print(f"Test CDMs: {len(df_test):,} across {df_test['event_id'].nunique():,} events. Columns: {len(df_test.columns)}")
print(f"Test Private Labels: {len(df_priv):,} rows across {df_priv['event_id'].nunique():,} events.")

assert len(df_train.columns) == 103, "Train dataset must have 103 columns"
assert len(df_test.columns) == 103, "Test dataset must have 103 columns"

## 2. Check time_to_tca Range & 2-Day Cutoff

In [3]:
print("Train time_to_tca min / max:", df_train["time_to_tca"].min(), df_train["time_to_tca"].max())
print("Test time_to_tca min / max:", df_test["time_to_tca"].min(), df_test["time_to_tca"].max())
print("Test private time_to_tca min / max:", df_priv["time_to_tca"].min(), df_priv["time_to_tca"].max())

assert df_test["time_to_tca"].min() >= 2.0, "Public test data must contain only pre-cutoff CDMs (>= 2.0 days)"
assert df_priv["time_to_tca"].max() <= 1.0, "Private test CDMs must all be within 1.0 day of TCA"

## 3. Verify Risk Scale, Floor Value & Class Imbalance

In [4]:
floor_val = df_train["risk"].min()
floor_rows = (df_train["risk"] == -30.0).sum()
print(f"Exact risk floor value: {floor_val}")
print(f"Total CDMs at risk floor (-30.0): {floor_rows:,} ({floor_rows / len(df_train) * 100:.2f}%)")

# Event-level target inspection
train_sorted = df_train.sort_values(["event_id", "time_to_tca"])
final_cdms = train_sorted.groupby("event_id").first()
pre_cutoff = df_train[df_train["time_to_tca"] >= 2.0]

valid_events = pre_cutoff["event_id"].unique()
dropped_events = set(final_cdms.index) - set(valid_events)

print(f"Total train events: {len(final_cdms):,}")
print(f"Valid train events (>= 1 pre-cutoff CDM): {len(valid_events):,}")
print(f"Dropped events (0 pre-cutoff CDMs): {len(dropped_events):,}")

final_risks = final_cdms.loc[list(valid_events), "risk"]
high_risk_count = (final_risks >= -6.0).sum()
operational_count = (final_risks >= -4.0).sum()
floor_events = (final_risks == -30.0).sum()

print(f"Valid events with final risk >= -6.0: {high_risk_count} ({high_risk_count / len(valid_events) * 100:.2f}%)")
print(f"Valid events with final risk >= -4.0: {operational_count} ({operational_count / len(valid_events) * 100:.2f}%)")
print(f"Valid events with final risk at floor (-30.0): {floor_events} ({floor_events / len(valid_events) * 100:.2f}%)")

## 4. Column Names Verification

In [5]:
required_cols = [
    "risk", "max_risk_estimate", "max_risk_scaling",
    "miss_distance", "relative_speed",
    "relative_position_r", "relative_position_t", "relative_position_n",
    "relative_velocity_r", "relative_velocity_t", "relative_velocity_n",
    "t_sigma_r", "c_sigma_r", "t_sigma_t", "c_sigma_t", "t_sigma_n", "c_sigma_n",
    "t_position_covariance_det", "c_position_covariance_det",
    "mission_id", "c_object_type", "t_rcs_estimate", "c_rcs_estimate",
    "F10", "F3M", "SSN", "AP"
]
for c in required_cols:
    assert c in df_train.columns, f"Missing required column: {c}"
print("All required columns verified present in dataset.")